# 04 — Bonus: Görev Atama (Task Assignment)

> Bu notebook workshop'ta anlatılmaz. Ev çalışması olarak incelemeniz için hazırlanmıştır.

**Öğrenme hedefleri:**
1. Binary ve sürekli değişkenleri aynı modelde kullanmak (MILP)
2. Min-max objective'i doğrusallaştırmayı öğrenmek
3. Gantt grafiği ile sonucu görselleştirmek

**Tahmini süre:** ~30-45 dakika

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                   'pulp>=2.8,<4'], check=True)
    for _k in list(sys.modules):
        if _k == 'pulp' or _k.startswith('pulp.'):
            del sys.modules[_k]
    !git clone -q https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import pandas as pd
import matplotlib.pyplot as plt
from utils.plotting import show_status, plot_gantt

---
## Problem Tanımı

8 görev 3 işlemciye atanacak.  
Her görevin her işlemcideki **çalışma süresi** farklı.  
**Amaç:** Tüm görevlerin biteceği en geç zamanı (**makespan, C_max**) minimize et.

Örnekler: paralel derleme, iş yükü dengeleme, bulut task scheduling.

In [ ]:
# Görev x İşlemci süresi tablosu (deterministik)
proc_times = {
    0: {0: 3, 1: 5, 2: 2},
    1: {0: 7, 1: 4, 2: 6},
    2: {0: 2, 1: 3, 2: 8},
    3: {0: 5, 1: 6, 2: 3},
    4: {0: 4, 1: 2, 2: 5},
    5: {0: 6, 1: 7, 2: 4},
    6: {0: 3, 1: 4, 2: 6},
    7: {0: 8, 1: 3, 2: 2},
}

jobs     = list(proc_times.keys())
machines = list(proc_times[0].keys())

df = pd.DataFrame(proc_times).T
df.index.name = "Görev"
df.columns = [f"İşlemci {m}" for m in machines]
print("Çalışma Süreleri Tablosu:")
print(df)

---
## Neden `min max` Doğrudan Yazılamaz?

ILP'de objective **doğrusal** olmak zorunda. `min max(...)` doğrusal değil.

**Çözüm: `C_max` yardımcı değişkeni**

- `minimize C_max`
- `Σ_j p[j,m] · x[j,m] ≤ C_max` (her m için)

`C_max` kısıtlar tarafından aşağıdan sıkıştırılıyor; minimize edilince makespan'e eşit olacak.

---
## Model ve Çözüm

In [ ]:
prob = pulp.LpProblem("task_assignment", pulp.LpMinimize)

x = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
     for j in jobs for m in machines}
C_max = pulp.LpVariable("C_max", lowBound=0, cat="Continuous")

prob += C_max
for j in jobs:
    prob += pulp.lpSum(x[j, m] for m in machines) == 1
for m in machines:
    prob += pulp.lpSum(proc_times[j][m] * x[j, m] for j in jobs) <= C_max

prob.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob)

cmax_val = pulp.value(C_max)
assignment = {j: m for j in jobs for m in machines if pulp.value(x[j, m]) > 0.5}

print(f"\nMakespan: {cmax_val:.1f}")
print("Atama:")
for j in sorted(assignment):
    m = assignment[j]
    print(f"  Görev {j} → İşlemci {m}  (süre: {proc_times[j][m]})")

In [ ]:
fig = plot_gantt(assignment, proc_times, num_machines=len(machines), c_max_val=cmax_val)
plt.show()

print("\nİşlemci yükleri:")
for m in machines:
    load = sum(proc_times[j][m] for j in jobs if assignment.get(j) == m)
    print(f"  İşlemci {m}: {load}")

---
## 🔧 Değiştir-Gözlemle Görevleri

### Görev 1 — 4. İşlemci Ekle

> Aşağıdaki tabloda 4. işlemcinin süreleri açıkça belirtilmiş.  
> Makespan nasıl değişir?

In [ ]:
# 4. işlemcinin çalışma süreleri (deterministik)
proc_times_4m = {
    0: {0: 3, 1: 5, 2: 2, 3: 4},
    1: {0: 7, 1: 4, 2: 6, 3: 3},
    2: {0: 2, 1: 3, 2: 8, 3: 5},
    3: {0: 5, 1: 6, 2: 3, 3: 2},
    4: {0: 4, 1: 2, 2: 5, 3: 6},
    5: {0: 6, 1: 7, 2: 4, 3: 3},
    6: {0: 3, 1: 4, 2: 6, 3: 5},
    7: {0: 8, 1: 3, 2: 2, 3: 4},
}
machines_4 = [0, 1, 2, 3]

prob_g1 = pulp.LpProblem("task_4m", pulp.LpMinimize)
x_g1 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
        for j in jobs for m in machines_4}
cmax_g1 = pulp.LpVariable("C_max", lowBound=0)

prob_g1 += cmax_g1
for j in jobs:
    prob_g1 += pulp.lpSum(x_g1[j, m] for m in machines_4) == 1
for m in machines_4:
    prob_g1 += pulp.lpSum(proc_times_4m[j][m] * x_g1[j, m] for j in jobs) <= cmax_g1

prob_g1.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g1)

cmax_g1_val = pulp.value(cmax_g1)
assign_g1 = {j: m for j in jobs for m in machines_4 if pulp.value(x_g1[j, m]) > 0.5}

print(f"3 işlemci makespan: {cmax_val:.1f}")
print(f"4 işlemci makespan: {cmax_g1_val:.1f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
for ax, (asgn, pt, nm, cv, n_m) in zip(axes, [
    (assignment,  proc_times,    "3 işlemci", cmax_val,   3),
    (assign_g1,   proc_times_4m, "4 işlemci", cmax_g1_val, 4),
]):
    machine_time = {m_: 0 for m_ in range(n_m)}
    from utils.plotting import PALETTE
    colors_l = [PALETTE["blue"], PALETTE["orange"], PALETTE["green"], PALETTE["sky"],
                PALETTE["vermil"], PALETTE["purple"], PALETTE["black"], PALETTE["gray"]]
    for j_ in sorted(asgn):
        m_ = asgn[j_]
        dur = pt[j_][m_]
        ax.barh(m_, dur, left=machine_time[m_], color=colors_l[j_ % len(colors_l)],
                edgecolor="white", height=0.55)
        ax.text(machine_time[m_] + dur/2, m_, f"G{j_}",
                ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        machine_time[m_] += dur
    ax.axvline(cv, color=PALETTE["red"], linestyle="--", lw=1.8)
    ax.text(cv + 0.1, n_m - 0.7, f"C_max={cv:.1f}", color=PALETTE["red"], fontsize=8)
    ax.set_yticks(range(n_m))
    ax.set_yticklabels([f"İşlemci {m_}" for m_ in range(n_m)], fontsize=8)
    ax.set_xlabel("Zaman"); ax.set_title(nm)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.suptitle("3 vs 4 İşlemci — Makespan Karşılaştırması", fontsize=11)
plt.tight_layout()
plt.show()

### Görev 2 — Yasak Atama

> Görev 0, İşlemci 2'de çalışamıyor. Bu kısıtı ekle.

In [ ]:
prob_g2 = pulp.LpProblem("task_forbidden", pulp.LpMinimize)
x_g2 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
        for j in jobs for m in machines}
cmax_g2 = pulp.LpVariable("C_max", lowBound=0)

prob_g2 += cmax_g2
for j in jobs:
    prob_g2 += pulp.lpSum(x_g2[j, m] for m in machines) == 1
for m in machines:
    prob_g2 += pulp.lpSum(proc_times[j][m] * x_g2[j, m] for j in jobs) <= cmax_g2
prob_g2 += x_g2[0, 2] == 0  # YASAK

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g2)
print(f"Yasak atama ile makespan: {pulp.value(cmax_g2):.1f}")
print(f"Orijinal makespan       : {cmax_val:.1f}")

### Görev 3 — Objective Değiştir: Makespan vs Toplam Süre

> Makespan yerine toplam işlemci süresini minimize et ve Gantt'ları yan yana karşılaştır.

In [ ]:
prob_g3 = pulp.LpProblem("task_total", pulp.LpMinimize)
x_g3 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
        for j in jobs for m in machines}

prob_g3 += pulp.lpSum(proc_times[j][m] * x_g3[j, m] for j in jobs for m in machines)
for j in jobs:
    prob_g3 += pulp.lpSum(x_g3[j, m] for m in machines) == 1

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))

assign_g3 = {j: m for j in jobs for m in machines if pulp.value(x_g3[j, m]) > 0.5}
total_v = int(pulp.value(prob_g3.objective))
makespan_g3 = max(
    sum(proc_times[j][m] for j in jobs if assign_g3.get(j) == m)
    for m in machines
)

print(f"Toplam süre minimize → toplam süre : {total_v}")
print(f"Bu çözümün makespan'i              : {makespan_g3}")
print(f"Orijinal makespan minimizasyonu    : {int(cmax_val)}")

# Yan yana Gantt
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=False)

for ax, (asgn, cv, title) in zip(axes, [
    (assignment, cmax_val,   f"Makespan Minimize  (C_max={cmax_val:.1f})"),
    (assign_g3,  makespan_g3, f"Toplam Süre Minimize  (C_max={makespan_g3})"),
]):
    from utils.plotting import PALETTE
    colors_l = [PALETTE["blue"], PALETTE["orange"], PALETTE["green"], PALETTE["sky"],
                PALETTE["vermil"], PALETTE["purple"], PALETTE["black"], PALETTE["gray"]]
    machine_time = {m_: 0 for m_ in machines}
    for j_ in sorted(asgn):
        m_ = asgn[j_]
        dur = proc_times[j_][m_]
        ax.barh(m_, dur, left=machine_time[m_], color=colors_l[j_ % len(colors_l)],
                edgecolor="white", height=0.55)
        ax.text(machine_time[m_] + dur/2, m_, f"G{j_}",
                ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        machine_time[m_] += dur
    ax.axvline(cv, color=PALETTE["red"], linestyle="--", lw=1.8)
    ax.text(cv + 0.1, len(machines) - 0.7, f"C_max={cv:.1f}",
            color=PALETTE["red"], fontsize=8)
    ax.set_yticks(machines)
    ax.set_yticklabels([f"İşlemci {m_}" for m_ in machines], fontsize=8)
    ax.set_xlabel("Zaman"); ax.set_title(title)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.suptitle("Objective Karşılaştırması: Makespan vs Toplam Süre", fontsize=11)
plt.tight_layout()
plt.show()

print()
print("Yorum: Toplam süreyi minimize etmek her görevi 'en hızlı işlemciye' iter,")
print("bu da o işlemciyi aşırı yükler ve makespan'i artırır.")